In [2]:
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

In [3]:
filename_jan = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-01.parquet"
filename_feb = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-02.parquet"

In [4]:
df = pd.read_parquet(filename_jan)

In [5]:
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,7.2,1.00,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,1.0,N,163,162,2,7.9,4.25,0.5,0.00,0.0,1.0,13.65,2.5,0.0,0.75
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0.0,1.40,1.0,N,43,237,1,10.7,4.25,0.5,2.50,0.0,1.0,18.95,2.5,0.0,0.75
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,38.7,1.00,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0.0,2.16,1.0,N,88,144,1,13.5,1.00,0.5,3.85,0.0,1.0,23.10,2.5,0.0,0.75


In [6]:
Q1 = df.shape

In [7]:
Q1

(3724889, 20)

In [8]:
df.tpep_dropoff_datetime = pd.to_datetime(df.tpep_dropoff_datetime)
df.tpep_pickup_datetime = pd.to_datetime(df.tpep_pickup_datetime)

df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime

df.duration = df.duration.apply( lambda td: td.total_seconds() / 60)

In [9]:
std = df['duration'].std()

In [10]:
Q2 = std

In [11]:
Q2

np.float64(25.17801981831615)

In [12]:
old_len = len(df)
df = df[((df.duration >= 1) & (df.duration <= 60))]
n_len = len(df)

In [13]:
Q3 = n_len / old_len

In [14]:
Q3

0.958934346768454

In [15]:
dict = DictVectorizer()

In [16]:
categorical = ['PULocationID', 'DOLocationID']
df[categorical] = df[categorical].astype(str)

train_dict = df[categorical].to_dict(orient='records')

In [17]:
train_dict

[{'PULocationID': '239', 'DOLocationID': '238'},
 {'PULocationID': '163', 'DOLocationID': '162'},
 {'PULocationID': '43', 'DOLocationID': '237'},
 {'PULocationID': '142', 'DOLocationID': '209'},
 {'PULocationID': '88', 'DOLocationID': '144'},
 {'PULocationID': '144', 'DOLocationID': '137'},
 {'PULocationID': '142', 'DOLocationID': '50'},
 {'PULocationID': '50', 'DOLocationID': '234'},
 {'PULocationID': '161', 'DOLocationID': '45'},
 {'PULocationID': '237', 'DOLocationID': '263'},
 {'PULocationID': '161', 'DOLocationID': '144'},
 {'PULocationID': '144', 'DOLocationID': '79'},
 {'PULocationID': '163', 'DOLocationID': '48'},
 {'PULocationID': '234', 'DOLocationID': '90'},
 {'PULocationID': '161', 'DOLocationID': '229'},
 {'PULocationID': '162', 'DOLocationID': '170'},
 {'PULocationID': '90', 'DOLocationID': '87'},
 {'PULocationID': '90', 'DOLocationID': '41'},
 {'PULocationID': '239', 'DOLocationID': '263'},
 {'PULocationID': '262', 'DOLocationID': '141'},
 {'PULocationID': '140', 'DOLoca

In [18]:
dict = DictVectorizer()
x_train = dict.fit_transform(train_dict)
y_train = df.duration.values

In [19]:
Q4 = x_train
Q4

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 7143848 stored elements and shape (3571924, 521)>

In [20]:
lr = LinearRegression()
lr.fit(x_train, y_train)
y_pred = lr.predict(x_train)
rmse = root_mean_squared_error(y_train, y_pred)

In [21]:
Q5 = rmse

In [22]:
Q5

8.989025860177977

In [23]:
df_val = pd.read_parquet(filename_feb)

In [24]:
df_val.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,7,2026-02-01 00:05:57,2026-02-01 00:05:57,1.0,0.94,1.0,N,107,170,1,7.2,0.00,0.5,0.00,0.0,1.0,12.95,2.5,0.00,0.75
1,7,2026-02-01 00:35:58,2026-02-01 00:35:58,1.0,1.93,1.0,N,234,141,1,11.4,0.00,0.5,3.43,0.0,1.0,20.58,2.5,0.00,0.75
2,2,2026-02-01 00:08:41,2026-02-01 00:39:32,1.0,9.99,1.0,N,138,68,1,44.3,6.00,0.5,11.01,0.0,1.0,67.81,2.5,1.75,0.75
3,1,2026-02-01 00:29:06,2026-02-01 00:41:04,0.0,1.70,1.0,N,209,13,1,12.8,4.25,0.5,3.70,0.0,1.0,22.25,2.5,0.00,0.75
4,1,2026-02-01 00:53:52,2026-02-01 01:11:21,0.0,3.70,1.0,N,249,229,1,19.8,4.25,0.5,6.35,0.0,1.0,31.90,2.5,0.00,0.75


In [25]:
df_val.tpep_dropoff_datetime = pd.to_datetime(df_val.tpep_dropoff_datetime)
df_val.tpep_pickup_datetime = pd.to_datetime(df_val.tpep_pickup_datetime)

df_val['duration'] = df_val.tpep_dropoff_datetime - df_val.tpep_pickup_datetime

df_val.duration = df_val.duration.apply( lambda td: td.total_seconds() / 60)

In [26]:
df_val.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,duration
0,7,2026-02-01 00:05:57,2026-02-01 00:05:57,1.0,0.94,1.0,N,107,170,1,...,0.00,0.5,0.00,0.0,1.0,12.95,2.5,0.00,0.75,0.000000
1,7,2026-02-01 00:35:58,2026-02-01 00:35:58,1.0,1.93,1.0,N,234,141,1,...,0.00,0.5,3.43,0.0,1.0,20.58,2.5,0.00,0.75,0.000000
2,2,2026-02-01 00:08:41,2026-02-01 00:39:32,1.0,9.99,1.0,N,138,68,1,...,6.00,0.5,11.01,0.0,1.0,67.81,2.5,1.75,0.75,30.850000
3,1,2026-02-01 00:29:06,2026-02-01 00:41:04,0.0,1.70,1.0,N,209,13,1,...,4.25,0.5,3.70,0.0,1.0,22.25,2.5,0.00,0.75,11.966667
4,1,2026-02-01 00:53:52,2026-02-01 01:11:21,0.0,3.70,1.0,N,249,229,1,...,4.25,0.5,6.35,0.0,1.0,31.90,2.5,0.00,0.75,17.483333


In [27]:
df_val = df_val[((df_val.duration >= 1) & (df_val.duration <= 60))]
categorical = ['PULocationID', 'DOLocationID']
df_val[categorical] = df_val[categorical].astype(str)

val_dicts = df_val[categorical].to_dict(orient='records')

In [28]:
val_dicts

[{'PULocationID': '138', 'DOLocationID': '68'},
 {'PULocationID': '209', 'DOLocationID': '13'},
 {'PULocationID': '249', 'DOLocationID': '229'},
 {'PULocationID': '113', 'DOLocationID': '90'},
 {'PULocationID': '234', 'DOLocationID': '144'},
 {'PULocationID': '237', 'DOLocationID': '151'},
 {'PULocationID': '148', 'DOLocationID': '263'},
 {'PULocationID': '79', 'DOLocationID': '170'},
 {'PULocationID': '161', 'DOLocationID': '114'},
 {'PULocationID': '263', 'DOLocationID': '116'},
 {'PULocationID': '237', 'DOLocationID': '141'},
 {'PULocationID': '181', 'DOLocationID': '49'},
 {'PULocationID': '97', 'DOLocationID': '181'},
 {'PULocationID': '114', 'DOLocationID': '114'},
 {'PULocationID': '114', 'DOLocationID': '249'},
 {'PULocationID': '249', 'DOLocationID': '158'},
 {'PULocationID': '68', 'DOLocationID': '162'},
 {'PULocationID': '114', 'DOLocationID': '209'},
 {'PULocationID': '170', 'DOLocationID': '170'},
 {'PULocationID': '132', 'DOLocationID': '260'},
 {'PULocationID': '239', 'D

In [29]:
x_val = dict.transform(val_dicts)
y_val = df_val.duration.values
y_pred = lr.predict(x_val)
rmse = root_mean_squared_error(y_val, y_pred)

In [30]:
rmse

9.289483087304598